<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN/blob/main/07_rag/information_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Information Retrieval: Lexical and Dense Search

Sentence embeddings gave us a way to measure semantic similarity. An information retrieval system turns that idea into a ranking problem: given a query and a collection of documents, which documents should we inspect first?

In this practice we will compare three approaches on the same small corpus. TF-IDF and BM25 use the words that appear in the query and documents; dense retrieval uses learned sentence embeddings. Holding the corpus and queries fixed will let us see where exact terms help, where paraphrases break lexical search and why a first-ranked document is not necessarily an answer.

In [1]:
# @title Setup
%pip install -q "requests==2.32.4" "scikit-learn==1.7.1" "rank-bm25==0.2.2" "sentence-transformers==5.2.0" "transformers==5.16.1"

import logging
import os
import warnings

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore", message=r"(?s).*HF_TOKEN.*")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("torchao").setLevel(logging.ERROR)
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()

## The corpus

We use a deliberately small corpus so that we can inspect every score and understand why a document was ranked. Documents D0 and D1 describe almost the same event with different vocabulary; the remaining documents provide nearby and unrelated topics.

This is useful for understanding the mechanisms, but it is not a realistic benchmark. Later we will add a small evaluation to make that limitation explicit.

In [2]:
documents = [
    "The company increased employee salaries after a strong quarter.",
    "The firm raised workers' pay following good results.",
    "The company opened a new office in Madrid.",
    "Employees requested flexible working hours.",
    "The ocean appears blue because water absorbs red light.",
    "A quick brown fox jumps over a lazy dog.",
]

for document_id, document in enumerate(documents):
    print(f"D{document_id}: {document}")

D0: The company increased employee salaries after a strong quarter.
D1: The firm raised workers' pay following good results.
D2: The company opened a new office in Madrid.
D3: Employees requested flexible working hours.
D4: The ocean appears blue because water absorbs red light.
D5: A quick brown fox jumps over a lazy dog.


## Tokenization for lexical retrieval

TF-IDF and BM25 work with terms. Use the same lowercase word tokenizer for documents and queries, and remove common English stopwords.

In [3]:
import re
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [word for word in words if word not in stopwords]

tokenized_documents = [tokenize(document) for document in documents]
for document_id, tokens in enumerate(tokenized_documents):
    print(f"D{document_id}: {tokens}")

D0: ['company', 'increased', 'employee', 'salaries', 'strong', 'quarter']
D1: ['firm', 'raised', 'workers', 'pay', 'following', 'good', 'results']
D2: ['company', 'opened', 'new', 'office', 'madrid']
D3: ['employees', 'requested', 'flexible', 'working', 'hours']
D4: ['ocean', 'appears', 'blue', 'water', 'absorbs', 'red', 'light']
D5: ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']


## TF-IDF and BM25

TF-IDF represents each document as a weighted term vector and compares it with the query using cosine similarity. BM25 is a ranking function developed specifically for search: it also rewards informative matching terms, but adds document-length normalization and limits the benefit of repeating the same term many times.

BM25 is a strong lexical baseline and remains widely useful in real search systems. Its scores and TF-IDF cosine scores live on different scales, so we will compare their rankings rather than compare the raw numbers directly.

In [4]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=tokenize, token_pattern=None, lowercase=False, norm="l2"
)
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
bm25 = BM25Okapi(tokenized_documents)

def tfidf_scores(query):
    query_vector = tfidf_vectorizer.transform([query])
    return cosine_similarity(query_vector, tfidf_matrix)[0]

def bm25_scores(query):
    return np.asarray(bm25.get_scores(tokenize(query)))

def top_indices(scores, k=3):
    return np.argsort(scores)[::-1][:k]

def print_ranking(scores, k=3):
    for rank, document_id in enumerate(top_indices(scores, k), start=1):
        print(f"{rank}. D{document_id} | score={scores[document_id]:.4f} | {documents[document_id]}")

### Exact terms

Start with a query that shares two terms with D0.

In [5]:
lexical_query = "employee salaries"
lexical_tfidf = tfidf_scores(lexical_query)
lexical_bm25 = bm25_scores(lexical_query)

print("TF-IDF")
print_ranking(lexical_tfidf)
print("\nBM25")
print_ranking(lexical_bm25)

TF-IDF
1. D0 | score=0.5938 | The company increased employee salaries after a strong quarter.
2. D5 | score=0.0000 | A quick brown fox jumps over a lazy dog.
3. D4 | score=0.0000 | The ocean appears blue because water absorbs red light.

BM25
1. D0 | score=2.5986 | The company increased employee salaries after a strong quarter.
2. D5 | score=0.0000 | A quick brown fox jumps over a lazy dog.
3. D4 | score=0.0000 | The ocean appears blue because water absorbs red light.


Both methods rank D0 first because it contains *employee* and *salaries*. D1 describes a similar event, but *workers* and *pay* do not match those query terms. Lexical retrieval is strong when the vocabulary overlaps.

### A paraphrase with no shared terms

Now ask for the same topic using *business*, *staff* and *compensation*.

In [6]:
paraphrase_query = "How did the business improve staff compensation?"
paraphrase_tfidf = tfidf_scores(paraphrase_query)
paraphrase_bm25 = bm25_scores(paraphrase_query)

print(f"Query tokens: {tokenize(paraphrase_query)}")
print(f"Maximum TF-IDF score: {paraphrase_tfidf.max():.4f}")
print(f"Maximum BM25 score:    {paraphrase_bm25.max():.4f}")
print("\nThe displayed order is only a tie between zero scores:")
print_ranking(paraphrase_tfidf)

Query tokens: ['did', 'business', 'improve', 'staff', 'compensation']
Maximum TF-IDF score: 0.0000
Maximum BM25 score:    0.0000

The displayed order is only a tie between zero scores:
1. D5 | score=0.0000 | A quick brown fox jumps over a lazy dog.
2. D4 | score=0.0000 | The ocean appears blue because water absorbs red light.
3. D3 | score=0.0000 | Employees requested flexible working hours.


Every lexical score is zero. The printed document order is not a meaningful ranking; it is just how the sorting code breaks a tie. This is more informative than saying the method returned a wrong top document: with this vocabulary, it had no matching signal at all.

## Dense retrieval

Lexical methods cannot match words they never see as related. Dense retrieval instead embeds both the query and each document with `all-MiniLM-L6-v2`, the same compact Sentence Transformer used in the semantics practice. We then compare those vectors with cosine similarity.

Using the same model makes the progression visible: a sentence-level similarity measure now becomes a document-ranking system.

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model_id = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_id)
document_embeddings = embedding_model.encode(documents, normalize_embeddings=True)

def dense_scores(query):
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    return document_embeddings @ query_embedding

paraphrase_dense = dense_scores(paraphrase_query)
print_ranking(paraphrase_dense)

1. D1 | score=0.5711 | The firm raised workers' pay following good results.
2. D0 | score=0.5499 | The company increased employee salaries after a strong quarter.
3. D3 | score=0.3175 | Employees requested flexible working hours.


Dense retrieval ranks D1 and D0 first even though the query shares no content words with them. The embedding model connects *business/company/firm*, *staff/workers/employee* and *compensation/pay/salaries*.

This is one small, constructed example. It shows a capability, not that dense retrieval is always better. Exact names, codes and rare terms can still favour lexical methods.

## What if the corpus has no answer?

A ranking function always has a first document. That does not mean the first document is relevant.

In [8]:
missing_query = "What is the CEO name?"
missing_tfidf = tfidf_scores(missing_query)
missing_dense = dense_scores(missing_query)

print("TF-IDF")
print_ranking(missing_tfidf, k=1)
print("\nDense")
print_ranking(missing_dense, k=1)

TF-IDF
1. D5 | score=0.0000 | A quick brown fox jumps over a lazy dog.

Dense
1. D1 | score=0.1989 | The firm raised workers' pay following good results.


TF-IDF returns a zero-score tie. Dense retrieval still returns D1 with a low positive score because it must rank something. Neither result contains a CEO name. Abstention requires a calibrated threshold or a later component that checks whether the evidence supports an answer.

## A small evaluation

Define relevant documents for three answerable queries and calculate Hit@2. This only checks whether at least one relevant document appears in the first two positions.

In [9]:
evaluation_queries = [
    ("employee salaries", {0, 1}),
    ("How did the business improve staff compensation?", {0, 1}),
    ("Why is the sea blue?", {4}),
]

retrievers = {
    "TF-IDF": tfidf_scores,
    "BM25": bm25_scores,
    "Dense": dense_scores,
}

hit_at_2 = {}
for name, score_function in retrievers.items():
    hits = []
    for query, relevant_ids in evaluation_queries:
        retrieved_ids = set(top_indices(score_function(query), k=2))
        hits.append(bool(retrieved_ids & relevant_ids))
    hit_at_2[name] = sum(hits) / len(hits)
    print(f"{name:<7} Hit@2 = {hit_at_2[name]:.3f} | per query: {hits}")

TF-IDF  Hit@2 = 0.667 | per query: [True, False, True]
BM25    Hit@2 = 0.667 | per query: [True, False, True]
Dense   Hit@2 = 1.000 | per query: [True, True, True]


Dense retrieval hits a relevant document for all three queries. TF-IDF and BM25 miss the paraphrase and score 2 out of 3.

The corpus and queries were created for this demonstration, so this is a code check rather than a benchmark. A real comparison needs many queries, relevance judgements and metrics such as Recall@k, MRR or nDCG.

# Takeaway

- TF-IDF and BM25 are strong when queries and documents share informative terms.
- Dense retrieval can connect paraphrases with little or no lexical overlap.
- The score scales of different retrieval methods are not directly comparable.
- A top-ranked document is not automatically relevant, especially when the answer is absent.
- Retrieval quality must be evaluated before adding a generator.

## Things to try

- Add a document containing an exact product code and compare lexical and dense retrieval.
- Remove stopword filtering and check which queries or rankings change.
- Create one query where BM25 beats dense retrieval and explain which signal made the difference.